# 1. Decode public EEG with an inspectable baseline
 **Question:** can six band-power features distinguish cued imagined left and right fist trials in one participant's recordings?

 Use PhysioNet EEGMMIDB **S001 R04/R08/R12**. Here T1 means imagined left fist, T2 imagined right fist, T0 instructed rest.
 The meaning changes in other runs. R04 trains, R08 validates, R12 remains the final test. These are **runs of one participant**, not cross-participant or cross-session validation.
 No EEG hardware, GPU, API key, flickering stimuli, or data collection is needed. Public EDFs download only when you run the loader.

In [ ]:
%pip -q install "numpy>=1.26,<3" "scipy>=1.11,<2" "mne>=1.8,<2" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4"

In [ ]:
from pathlib import Path
import hashlib, sys, urllib.request
SOURCE_COMMIT = "22dcdced86cc425a2d50ffa8516fe9581730ebc9"
EXPECTED_SHA256 = "90c8c015e114b3b9a89bb4206dbc4f6094969a792b1a0263e0009e673f935567"
candidates = [Path("bci_studio.py"), Path("chapters/ch20-bci-foundations/bci_studio.py")]
helper = next((p for p in candidates if p.is_file()), None)
if helper is None:
    target = Path("chapter20_runtime")
    target.mkdir(exist_ok=True)
    helper = target / "bci_studio.py"
    url = f"https://raw.githubusercontent.com/ZagElSayed/Introduction-to-Neuroinformatics/{SOURCE_COMMIT}/chapters/ch20-bci-foundations/bci_studio.py"
    payload = urllib.request.urlopen(url, timeout=60).read()
    if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
        raise RuntimeError("Downloaded helper checksum mismatch")
    helper.write_bytes(payload)
sys.path.insert(0, str(helper.resolve().parent))
print("Helper:", helper, "SHA-256:", hashlib.sha256(helper.read_bytes()).hexdigest())
import bci_studio as bci

## Load the contract and build the baseline
 Channels are C3, Cz, C4 in that order, at 160 Hz, in volts as returned by MNE. The helper rejects a changed contract.
 Each run is filtered causally with zero initial state. One [.5, 2.5)-s window per T1/T2 cue supplies six log10 band-power features (8–12 and 13–30 Hz for each channel).
 StandardScaler and logistic regression are fitted on R04 only. Parameters are preset for teaching, not optimized for a high score.

In [ ]:
experiment = bci.build_experiment()
cfg = experiment["config"]
for role, values in experiment["manifest"]["scores"].items():
    print(role, values)
print("Versions:", experiment["manifest"]["versions"])
print("Source files:", experiment["manifest"]["sources"])

In [ ]:
import matplotlib.pyplot as plt
raw = experiment["runs"][cfg.train_run]["data"]
t = __import__("numpy").arange(int(6 * cfg.fs)) / cfg.fs
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, raw[0, :len(t)] * 1e6, color="#167D91")
ax.set(xlabel="Recorded time (s)", ylabel="C3 (µV)", title="Recorded EEG: inspect before decoding")
plt.show()

## What do the numbers establish?
 Training performance is descriptive. Validation performance is a small within-participant/run-transfer estimate; trial counts are limited and individual performance may be poor.
 A correct label is a **cue-associated class**, not a direct measurement of intent. Artifact confounding remains possible. Simple amplitude/flatness checks do not validate physiological artifact removal.
 Keep the test run unopened for model or policy selection. Use a new held-out run or participant if you select changes after seeing test outcomes.

In [ ]:
receipt = bci.save_bundle({"kind": "training_validation_manifest", "manifest": experiment["manifest"]})
print(receipt)

## Reflection
 Explain why a random split of neighboring replay windows would answer a weaker question. Which field in the manifest prevents a left/right label from silently becoming a hands/feet label? What does a file hash establish, and what does it leave unproven?
 Continue with **02_replay_and_command_policy.ipynb**. No test accuracy is claimed in this notebook.